# FashionCLIP Metric Learning (triplet pipeline)

Этот ноутбук делает **дообучение только image-энкодера FashionCLIP** в парадигме metric learning.


In [ ]:
import os, cv2, math, time, random
import numpy as np
import pandas as pd
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor, CLIPModel
from sklearn.metrics import roc_auc_score, average_precision_score


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

@dataclass
class Config:
    triplets_train_path: str = "data/train_triplets.parquet"
    triplets_val_path: str = "data/val_triplets.parquet"
    val_pairs_path: Optional[str] = "data/val_pairs.parquet"
    hf_model_name: str = "patrickjohncyh/fashion-clip"
    output_dir: str = "artifacts/fashion_clip_metric"
    batch_size: int = 64
    num_workers: int = 8
    prefetch_factor: int = 4
    persistent_workers: bool = True
    lr: float = 2e-5
    weight_decay: float = 1e-4
    epochs: int = 6
    trainable_vision_layers: int = 4
    use_projection_head: bool = False
    projection_dim: int = 512
    loss_names: Tuple[str, ...] = ("triplet", "softplus_triplet", "circle")
    margin: float = 0.2
    gamma: float = 80.0
    circle_m: float = 0.25
    use_amp: bool = True
    max_grad_norm: float = 1.0

cfg = Config()
os.makedirs(cfg.output_dir, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(cfg)
print("device:", device)


In [ ]:
def load_table(path: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(path)
    if p.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path)

train_triplets = load_table(cfg.triplets_train_path)
val_triplets = load_table(cfg.triplets_val_path)


In [ ]:
def collect_unique_paths(*dfs: pd.DataFrame) -> List[str]:
    paths = set()
    for df in dfs:
        for c in ["anchor_path", "pos_path", "neg_path"]:
            if c in df.columns:
                paths.update(df[c].astype(str).tolist())
    return sorted(paths)


def build_image_cache(paths: List[str], verbose: bool = True) -> Dict[str, np.ndarray]:
    """
    Простой RAM-cache как в базовом ноутбуке:
    IMAGE_CACHE = {"/path/to/image.jpg": np.ndarray(H, W, 3) в RGB}
    """
    cache = {}
    it = tqdm(paths, desc="Caching images") if verbose else paths
    for p in it:
        img_bgr = cv2.imread(str(p), cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise FileNotFoundError(p)
        cache[str(p)] = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    total_mb = sum(v.nbytes for v in cache.values()) / (1024**2)
    print(f"Cached images: {len(cache)} | approx RAM: {total_mb:.1f} MB")
    return cache


all_paths = collect_unique_paths(train_triplets, val_triplets)
IMAGE_CACHE = build_image_cache(all_paths, verbose=True)


In [ ]:
class TripletImageDataset(Dataset):
    def __init__(self, triplet_df: pd.DataFrame, image_cache: Dict[str, np.ndarray]):
        self.df = triplet_df.reset_index(drop=True)
        self.image_cache = image_cache
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return self.image_cache[str(row.anchor_path)], self.image_cache[str(row.pos_path)], self.image_cache[str(row.neg_path)]

def make_collate(processor: CLIPProcessor):
    def collate_fn(batch):
        a_imgs, p_imgs, n_imgs = zip(*batch)
        a = processor(images=list(a_imgs), return_tensors="pt")["pixel_values"]
        p = processor(images=list(p_imgs), return_tensors="pt")["pixel_values"]
        n = processor(images=list(n_imgs), return_tensors="pt")["pixel_values"]
        return a, p, n
    return collate_fn

processor = CLIPProcessor.from_pretrained(cfg.hf_model_name)
collate_fn = make_collate(processor)
train_loader = DataLoader(TripletImageDataset(train_triplets, IMAGE_CACHE), batch_size=cfg.batch_size, shuffle=True, drop_last=True, num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available(), collate_fn=collate_fn)
val_loader = DataLoader(TripletImageDataset(val_triplets, IMAGE_CACHE), batch_size=cfg.batch_size, shuffle=False, drop_last=False, num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available(), collate_fn=collate_fn)


In [ ]:
class FashionCLIPMetricModel(nn.Module):
    def __init__(self, model_name: str, trainable_vision_layers: int = 4, use_projection_head: bool = False, projection_dim: int = 512):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(model_name)

        # 1) Сначала замораживаем все веса CLIP
        for p in self.clip.parameters():
            p.requires_grad = False

        # 2) Размораживаем visual projection (проекция image-фич в embedding space)
        for p in self.clip.visual_projection.parameters():
            p.requires_grad = True

        # 3) Размораживаем последние N блоков visual encoder
        layers = self.clip.vision_model.encoder.layers
        for i in range(max(0, len(layers)-trainable_vision_layers), len(layers)):
            for p in layers[i].parameters():
                p.requires_grad = True

        # 4) Опциональная дополнительная проекционная голова
        emb_dim = self.clip.config.projection_dim
        self.head = nn.Identity() if not use_projection_head else nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(emb_dim, projection_dim)
        )

    def _image_features_compat(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """
        Совместимый путь для разных версий transformers:
        - в одних версиях get_image_features() возвращает Tensor,
        - в других окружениях может прилетать объект/иной тип.

        Поэтому берём фичи явно через vision_model + visual_projection.
        """
        vision_out = self.clip.vision_model(pixel_values=pixel_values)
        pooled = vision_out.pooler_output
        image_features = self.clip.visual_projection(pooled)
        return image_features

    def encode(self, pixel_values: torch.Tensor) -> torch.Tensor:
        z = self._image_features_compat(pixel_values)
        z = self.head(z)
        return F.normalize(z, p=2, dim=-1)

    def forward(self, a_px, p_px, n_px):
        return self.encode(a_px), self.encode(p_px), self.encode(n_px)


## Подробный разбор класса `FashionCLIPMetricModel` (построчно)

Ниже объяснение **каждой важной строки** класса, чтобы было понятно, что и зачем дообучается.

- `self.clip = CLIPModel.from_pretrained(model_name)` — загружаем готовый FashionCLIP.
- `for p in self.clip.parameters(): p.requires_grad = False` — сначала замораживаем **все** параметры, чтобы контролировать, какие части учим.
- `for p in self.clip.visual_projection.parameters(): p.requires_grad = True` — включаем обучение проекционной головы image-ветки, т.к. она напрямую формирует embedding-space.
- `layers = self.clip.vision_model.encoder.layers` — берём список transformer-блоков визуального энкодера.
- `for i in range(max(0, len(layers)-trainable_vision_layers), len(layers)):` — выбираем последние `N` слоёв (они отвечают за более высокоуровневые признаки и обычно лучше адаптируются к новой задаче).
- `for p in layers[i].parameters(): p.requires_grad = True` — размораживаем только эти последние блоки.
- `emb_dim = self.clip.config.projection_dim` — размерность исходного CLIP-эмбеддинга.
- `self.head = nn.Identity() ...` — опциональный projection head:
  - `Identity`, если не хотим дополнительной MLP;
  - `Linear -> GELU -> Dropout -> Linear`, если хотим дополнительную адаптацию embedding-space.

### Важный compatibility-метод `_image_features_compat`
- Мы **не полагаемся напрямую** на `get_image_features()`, так как в разных версиях/сборках `transformers` его поведение может отличаться.
- Вместо этого явно делаем:
  1) `vision_out = self.clip.vision_model(pixel_values=...)`
  2) `pooled = vision_out.pooler_output`
  3) `image_features = self.clip.visual_projection(pooled)`
- Такой путь стабилен и возвращает именно `Tensor`, что устраняет ошибку вида: `BaseModelOutputWithPooling has no attribute norm`.

### Метод `encode`
- Берёт Tensor фичей из `_image_features_compat`.
- Прогоняет через `self.head`.
- Делает `F.normalize(z, p=2, dim=-1)`, чтобы cosine similarity считался просто скалярным произведением.

### Метод `forward`
- Возвращает `(za, zp, zn)` — эмбеддинги якоря, позитива, негатива для triplet-обучения.


In [ ]:
def pairwise_cos(a, b):
    return (F.normalize(a, dim=-1) * F.normalize(b, dim=-1)).sum(dim=-1)

def triplet_loss(za, zp, zn, margin=0.2):
    return F.triplet_margin_loss(za, zp, zn, margin=margin, p=2)

def softplus_triplet_loss(za, zp, zn):
    d_ap = torch.norm(za - zp, p=2, dim=-1)
    d_an = torch.norm(za - zn, p=2, dim=-1)
    return F.softplus(d_ap - d_an).mean()

def circle_triplet_loss(za, zp, zn, m=0.25, gamma=80.0):
    s_p = pairwise_cos(za, zp)
    s_n = pairwise_cos(za, zn)
    alpha_p = torch.relu(1 + m - s_p)
    alpha_n = torch.relu(s_n + m)
    logits = gamma * (alpha_n * (s_n - m) - alpha_p * (s_p - (1 - m)))
    return F.softplus(logits).mean()

def compute_loss(loss_name, za, zp, zn, cfg):
    if loss_name == "triplet":
        return triplet_loss(za, zp, zn, margin=cfg.margin)
    if loss_name == "softplus_triplet":
        return softplus_triplet_loss(za, zp, zn)
    if loss_name == "circle":
        return circle_triplet_loss(za, zp, zn, m=cfg.circle_m, gamma=cfg.gamma)
    raise ValueError(loss_name)


## Подробно про каждый loss: идея, плюсы и минусы

### 1) `triplet` (Triplet Margin Loss)
**Что делает:**
- Минимизирует `d(a,p)` и максимизирует `d(a,n)` с условием зазора `margin`.
- Формально: `max(0, d(a,p) - d(a,n) + margin)`.

**Почему хорош:**
- Простой, понятный, интерпретируемый.
- Хороший baseline для metric learning.
- Часто стабилен на старте, особенно при нормализованных эмбеддингах.

**Почему может быть плох:**
- У многих триплетов loss быстро становится 0 (лёгкие примеры перестают учить модель).
- Сильно зависит от качества negative sampling (без hard negatives прогресс может быть медленным).

### 2) `softplus_triplet`
**Что делает:**
- Использует гладкую версию triplet-идеи: `softplus(d(a,p) - d(a,n))`.
- Вместо жёсткого порога как в margin-формуле, штраф изменяется плавно.

**Почему хорош:**
- Более гладкие градиенты, чем у hinge-like triplet.
- Меньше риск «ступенчатого» обучения, когда много нулевых градиентов.

**Почему может быть плох:**
- Может медленнее формировать чёткий margin между классами.
- Иногда требует аккуратного тюнинга LR/батча для лучшей сепарации.

### 3) `circle` (Circle Loss для triplet-случая)
**Что делает:**
- Работает в cosine-space и вводит веса `alpha_p` / `alpha_n`, которые усиливают вклад «трудных» пар.
- Если позитив недостаточно близок или негатив слишком близок, штраф растёт сильнее.

**Почему хорош:**
- Лучше фокусируется на сложных примерах, а не усредняет всё одинаково.
- Часто даёт более компактные кластеры позитивов и лучшую разделимость с негативами.
- Хорошо сочетается с cosine-инференсом, т.к. оптимизирует похожесть напрямую.

**Почему может быть плох:**
- Более чувствителен к гиперпараметрам (`gamma`, `m`).
- Может быть менее стабилен без нормализации эмбеддингов и адекватного батча.
- Труднее интерпретировать и отлаживать, чем классический triplet.

### Практический совет по выбору
- Начать с `triplet` как baseline.
- Если обучение выходит на плато или мало hard-negatives — попробовать `softplus_triplet`.
- Если нужна максимальная разделимость и готовы тюнить гиперпараметры — `circle`.


In [ ]:
@torch.no_grad()
def triplet_batch_acc(za, zp, zn):
    return (pairwise_cos(za, zp) > pairwise_cos(za, zn)).float().mean().item()

def run_epoch(model, loader, optimizer, scaler, loss_name, train):
    model.train(train)
    losses, accs = [], []
    for a_px, p_px, n_px in tqdm(loader, leave=False):
        a_px, p_px, n_px = a_px.to(device), p_px.to(device), n_px.to(device)
        with torch.set_grad_enabled(train):
            with torch.cuda.amp.autocast(enabled=(cfg.use_amp and device.type == "cuda")):
                za, zp, zn = model(a_px, p_px, n_px)
                loss = compute_loss(loss_name, za, zp, zn, cfg)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
        losses.append(loss.item())
        accs.append(triplet_batch_acc(za.detach(), zp.detach(), zn.detach()))
    return {"loss": float(np.mean(losses)), "triplet_acc": float(np.mean(accs))}


In [ ]:
all_histories, best_ckpts = {}, {}
for loss_name in cfg.loss_names:
    model = FashionCLIPMetricModel(cfg.hf_model_name, cfg.trainable_vision_layers, cfg.use_projection_head, cfg.projection_dim).to(device)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and device.type == "cuda"))
    history, best_val = [], float("inf")
    best_path = os.path.join(cfg.output_dir, f"best_{loss_name}.pt")

    for epoch in range(1, cfg.epochs + 1):
        tr = run_epoch(model, train_loader, optimizer, scaler, loss_name=loss_name, train=True)
        va = run_epoch(model, val_loader, optimizer, scaler, loss_name=loss_name, train=False)
        history.append({"loss_name": loss_name, "epoch": epoch, "train_loss": tr["loss"], "val_loss": va["loss"], "train_triplet_acc": tr["triplet_acc"], "val_triplet_acc": va["triplet_acc"]})
        print(f"[{loss_name}] E{epoch:02d} tr_loss={tr['loss']:.4f} va_loss={va['loss']:.4f} tr_acc={tr['triplet_acc']:.4f} va_acc={va['triplet_acc']:.4f}")
        if va["loss"] < best_val:
            best_val = va["loss"]
            torch.save({"model_state": model.state_dict(), "config": cfg.__dict__, "loss_name": loss_name}, best_path)
    all_histories[loss_name] = pd.DataFrame(history)
    best_ckpts[loss_name] = best_path


In [ ]:
def pick_best_loss_by_val_triplet_acc(histories):
    best_name, best_metric = None, -1.0
    for name, h in histories.items():
        metric = float(h["val_triplet_acc"].max())
        if metric > best_metric:
            best_name, best_metric = name, metric
    return best_name

best_loss_name = pick_best_loss_by_val_triplet_acc(all_histories)
best_model = FashionCLIPMetricModel(cfg.hf_model_name, cfg.trainable_vision_layers, cfg.use_projection_head, cfg.projection_dim).to(device)
best_model.load_state_dict(torch.load(best_ckpts[best_loss_name], map_location=device)["model_state"])
best_model.eval()

@torch.no_grad()
def encode_image_paths(model, processor, image_paths, image_cache, batch_size=256):
    model.eval()
    vectors = []
    for i in tqdm(range(0, len(image_paths), batch_size), leave=False):
        chunk = image_paths[i:i+batch_size]
        imgs = [image_cache[str(p)] for p in chunk]
        px = processor(images=imgs, return_tensors="pt")["pixel_values"].to(device)
        vectors.append(model.encode(px).cpu())
    return torch.cat(vectors, dim=0)


## Инференс

1) Получаем эмбеддинги товаров через `encode_image_paths`.  
2) Считаем cosine: `(z_i * z_j).sum()`, так как векторы нормированы.  
3) Порог подбираем по валидации.

В этом ноутбуке реализованы 3 варианта лосса (включая **Circle Loss**) и переиспользован triplet-пайплайн с RAM-кешем изображений.


## Подробные пояснения по архитектуре и обучению

### Что именно дообучаем
- Основа — `patrickjohncyh/fashion-clip`.
- Обучаем только image-ветку: `visual_projection` и последние блоки `vision_model.encoder.layers`.
- Text-часть CLIP не используется, чтобы задача оставалась чисто image-only.

### Почему cosine на инференсе
- Модель возвращает L2-нормированные векторы, поэтому скалярное произведение эквивалентно cosine similarity.
- Это удобно для продакшна: эмбеддинги айтемов можно посчитать офлайн один раз, а затем быстро сравнивать любые пары.

### Лоссы
1. **Triplet margin loss** — базовый стабильный старт.
2. **Softplus-triplet** — более гладкий вариант без жёсткого margin-обрезания.
3. **Circle loss** — управляет вкладом сложных/лёгких примеров через `alpha_p/alpha_n`, часто даёт более чёткое разделение в embedding-space.

### Кэширование изображений
- Все уникальные пути из triplets поднимаются в RAM (`IMAGE_CACHE`), чтобы убрать bottleneck чтения диска.
- Кэш хранится как обычный словарь в RAM в течение сессии (`IMAGE_CACHE`).

### Как использовать результат
1. Выбрать лучший чекпоинт по `val_triplet_acc` (или по pair-level AUC, если есть `val_pairs`).
2. Посчитать эмбеддинги каталога.
3. Для пары айтемов считать cosine.
4. Порог бинарного решения подобрать на валидации под бизнес-метрику (precision/recall/F1, либо cost-aware метрика).

### Идеи для следующего шага
- Hard negative mining (online/offline).
- Увеличить разнообразие позитивов по стилям/категориям.
- Добавить TTA на инференсе и/или EMA весов при обучении.
- Калибровать порог отдельно для разных категорий одежды.


## Инференс на `test` для всех 3 моделей

Теперь добавим отдельный блок для вашего сценария:
- есть `test` с колонками `sku1`, `sku2`, `sku1_path`, `sku2_path`;
- нужно отскорить **каждой** моделью (`triplet`, `softplus_triplet`, `circle`).

Скор = cosine similarity между эмбеддингами пары.


In [ ]:
# Путь к test (parquet/csv)
cfg_test_path = "data/test.parquet"

def score_pairs_cosine_no_label(model, processor, pair_df: pd.DataFrame, image_cache: Dict[str, np.ndarray], batch_size: int = 256):
    required = {"sku1", "sku2", "sku1_path", "sku2_path"}
    miss = required - set(pair_df.columns)
    if miss:
        raise ValueError(f"test df missing columns: {miss}")

    model.eval()
    scores = []

    rows = pair_df[["sku1_path", "sku2_path"]].astype(str).values.tolist()
    for i in tqdm(range(0, len(rows), batch_size), desc="Scoring test", leave=False):
        chunk = rows[i:i+batch_size]

        imgs1, imgs2 = [], []
        for p1, p2 in chunk:
            if p1 not in image_cache or p2 not in image_cache:
                raise KeyError(f"Path not found in IMAGE_CACHE: {p1} | {p2}")
            imgs1.append(image_cache[p1])
            imgs2.append(image_cache[p2])

        px1 = processor(images=imgs1, return_tensors="pt")["pixel_values"].to(device)
        px2 = processor(images=imgs2, return_tensors="pt")["pixel_values"].to(device)

        z1 = model.encode(px1)
        z2 = model.encode(px2)
        s = (z1 * z2).sum(dim=-1)  # cosine, т.к. эмбеддинги L2-normalized
        scores.extend(s.detach().cpu().numpy().tolist())

    return np.array(scores, dtype=np.float32)


def ensure_test_paths_in_cache(test_df: pd.DataFrame, image_cache: Dict[str, np.ndarray]):
    needed = set(test_df["sku1_path"].astype(str)).union(set(test_df["sku2_path"].astype(str)))
    missing = sorted([p for p in needed if p not in image_cache])
    if missing:
        print(f"Adding {len(missing)} missing test images to IMAGE_CACHE...")
        image_cache.update(build_image_cache(missing, verbose=True))


def load_model_from_ckpt(loss_name: str):
    model = FashionCLIPMetricModel(
        cfg.hf_model_name,
        cfg.trainable_vision_layers,
        cfg.use_projection_head,
        cfg.projection_dim,
    ).to(device)
    ckpt_path = best_ckpts[loss_name]
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state["model_state"])
    model.eval()
    return model

# 1) Загружаем test
if not Path(cfg_test_path).exists():
    raise FileNotFoundError(f"Test file not found: {cfg_test_path}")

test_df = load_table(cfg_test_path)
print("test shape:", test_df.shape)

# 2) Убеждаемся, что все test-картинки есть в IMAGE_CACHE
ensure_test_paths_in_cache(test_df, IMAGE_CACHE)

# 3) Скорим всеми 3 моделями
losses_to_score = ["triplet", "softplus_triplet", "circle"]
for loss_name in losses_to_score:
    if loss_name not in best_ckpts:
        raise KeyError(f"Checkpoint for '{loss_name}' not found in best_ckpts. Available: {list(best_ckpts.keys())}")

    m = load_model_from_ckpt(loss_name)
    test_df[f"score_{loss_name}"] = score_pairs_cosine_no_label(m, processor, test_df, IMAGE_CACHE, batch_size=cfg.batch_size)

# 4) Простая агрегация (по желанию): средний скор по 3 моделям
test_df["score_mean_3models"] = test_df[["score_triplet", "score_softplus_triplet", "score_circle"]].mean(axis=1)

# 5) Сохраняем результат
out_path = os.path.join(cfg.output_dir, "test_scored_all_losses.parquet")
test_df.to_parquet(out_path, index=False)
print("Saved scored test to:", out_path)

test_df[["sku1", "sku2", "score_triplet", "score_softplus_triplet", "score_circle", "score_mean_3models"]].head()
